In [2]:
from dotenv import load_dotenv
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

In [3]:
load_dotenv()

db_user = os.getenv('DB_USER')
db_pass = os.getenv('DB_PASS')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')

engine = create_engine(f"postgresql+psycopg2://{db_user}:{db_pass}@{db_host}:{db_port}/{db_name}")

In [4]:
# Helper function to sort rounds
def get_round_order(row):
    # Match regular rounds: 'R1', 'R23', etc.
    match = re.match(r'R(\d+)$', row['round'])
    if match:
        return int(match.group(1))
    # Finals rounds mapping (AFL convention)
    finals_order = {
        'REF': 100,  # Elimination Final (week 1)
        'RQF': 101,  # Qualifying Final (week 1)
        'RSF': 102,  # Semi Final (week 2)
        'RPF': 103,  # Preliminary Final (week 3)
        'RGF': 104,  # Grand Final (week 4)
    }
    code = row['round'][1:] 
    return finals_order.get(code, 999)  # Unknown finals get 999

In [5]:
# dataframe for games
df_games = pd.read_sql("SELECT * FROM games", engine)
print(df_games.head())

      id round   game_date  venue_id  home_team_id  away_team_id  season_year  \
0  13358    R3  2010-04-09        14         20709         20158         2010   
1  13359    R3  2010-04-10        14         20525         20894         2010   
2  13360    R3  2010-04-10      1323         20893             5         2010   
3  13361    R3  2010-04-10         4         19974         20434         2010   
4  13362    R3  2010-04-10         1         19882         20065         2010   

   afltables_game_id home_score_str away_score_str  ...  kaggle_start_time  \
0              13594        10.9.69        4.17.41  ...               None   
1              13595      17.14.116       13.13.91  ...               None   
2              13596       11.15.81      16.12.108  ...               None   
3              13597       10.15.75       13.17.95  ...               None   
4              13598      16.15.111        7.14.56  ...               None   

   kaggle_attendance  kaggle_home_qt_score  

In [6]:
# function to add h2h history

def add_h2h_features(df_games, x=5, venue_specific=False):
    """
    Adds head-to-head features to the games dataframe.
    For each game, computes home team's wins and avg margin in last X games vs same opponent.
    """
    # Ensure games are sorted chronologically
    df_games = df_games.sort_values('game_date').reset_index(drop=True)
    features = []
    
    for idx, row in df_games.iterrows():
        home = row['home_team_id']
        away = row['away_team_id']
        game_date = row['game_date']
        venue = row['venue_id']
        
        # Find previous games between these two teams (either home/away order)
        h2h_mask = (
            (
                ((df_games['home_team_id'] == home) & (df_games['away_team_id'] == away)) |
                ((df_games['home_team_id'] == away) & (df_games['away_team_id'] == home))
            )
            & (df_games['game_date'] < game_date)
        )
        if venue_specific:
            h2h_mask &= (df_games['venue_id'] == venue)
        
        h2h_games = df_games[h2h_mask].sort_values('game_date', ascending=False).head(x)
        
        # Calculate home team's wins and average margin
        home_wins = 0
        margins = []
        for _, g in h2h_games.iterrows():
            if g['home_team_id'] == home:
                margin = g['home_score_total'] - g['away_score_total']
                if margin > 0:
                    home_wins += 1
                margins.append(margin)
            elif g['away_team_id'] == home:
                margin = g['away_score_total'] - g['home_score_total']
                if margin > 0:
                    home_wins += 1
                margins.append(margin)
        
        last_winner = None
        if not h2h_games.empty:
            last_game = h2h_games.iloc[0]
            if last_game['home_team_id'] == home:
                last_winner = int(last_game['home_score_total'] > last_game['away_score_total'])
            elif last_game['away_team_id'] == home:
                last_winner = int(last_game['away_score_total'] > last_game['home_score_total'])
        else:
            last_winner = None  # or set to -1
        
        features.append({
            'id': row['id'],
            f'h2h_wins_last{x}': home_wins,
            f'h2h_margin_last{x}': sum(margins)/len(margins) if margins else 0,
            f'h2h_count_last{x}': len(h2h_games),
            f'h2h_last_winner': last_winner if last_winner is not None else 0
        })
        
    df_features = pd.DataFrame(features)
    df_games = df_games.merge(df_features, on='id', how='left')
    return df_games

In [7]:
# Add in results for last however many games
games = df_games.copy()
for x in [3, 4, 5, 6, 7]:
    h2h_feats = add_h2h_features(df_games, x=x, venue_specific=False)[['id', 
        f'h2h_wins_last{x}', f'h2h_margin_last{x}', f'h2h_count_last{x}', f'h2h_last_winner']]
    # Rename the columns to be unique for this X
    h2h_feats = h2h_feats.rename(columns={
        f'h2h_wins_last{x}': f'h2h_wins_last{x}',
        f'h2h_margin_last{x}': f'h2h_margin_last{x}',
        f'h2h_count_last{x}': f'h2h_count_last{x}',
        f'h2h_last_winner': f'h2h_last_winner_{x}'
    })
    games = games.merge(h2h_feats, on='id', how='left')

KeyboardInterrupt: 

In [ ]:
# Evaluate which model is best
for x in [3, 4, 5, 6, 7]:
    games[f'h2h_winrate_last{x}'] = games[f'h2h_wins_last{x}'] / games[f'h2h_count_last{x}'].replace(0, np.nan)
    games[f'h2h_winrate_last{x}'] = games[f'h2h_winrate_last{x}'].fillna(0)
for x in [3, 4, 5, 6, 7]:
    feature = f'h2h_winrate_last{x}'
    X = games[[feature]].fillna(0)
    X = sm.add_constant(X)
    y = (games['home_result'] == 'W').astype(int)  # 1 = home win, 0 = not
    model = sm.Logit(y, X).fit(disp=0)
    print(f'X={x}: {feature}, coef={model.params[feature]:.3f}, p={model.pvalues[feature]:.3f}, AUC={model.prsquared:.3f}')

X=3: h2h_winrate_last3, coef=1.176, p=0.000, AUC=0.030
X=4: h2h_winrate_last4, coef=1.166, p=0.000, AUC=0.026
X=5: h2h_winrate_last5, coef=1.214, p=0.000, AUC=0.026
X=6: h2h_winrate_last6, coef=1.242, p=0.000, AUC=0.025
X=7: h2h_winrate_last7, coef=1.206, p=0.000, AUC=0.022


In [9]:
print(games.columns.tolist())

['id', 'round', 'game_date', 'venue_id', 'home_team_id', 'away_team_id', 'season_year', 'afltables_game_id', 'home_score_str', 'away_score_str', 'home_score_goals', 'home_score_behinds', 'home_score_total', 'away_score_goals', 'away_score_behinds', 'away_score_total', 'margin', 'home_result', 'away_result', 'max_temp', 'min_temp', 'rainfall', 'kaggle_data', 'kaggle_start_time', 'kaggle_attendance', 'kaggle_home_qt_score', 'kaggle_home_ht_score', 'kaggle_home_3qt_score', 'kaggle_home_ft_score', 'kaggle_away_qt_score', 'kaggle_away_ht_score', 'kaggle_away_3qt_score', 'kaggle_away_ft_score', 'h2h_wins_last3', 'h2h_margin_last3', 'h2h_count_last3', 'h2h_last_winner_3', 'h2h_wins_last4', 'h2h_margin_last4', 'h2h_count_last4', 'h2h_last_winner_4', 'h2h_wins_last5', 'h2h_margin_last5', 'h2h_count_last5', 'h2h_last_winner_5', 'h2h_wins_last6', 'h2h_margin_last6', 'h2h_count_last6', 'h2h_last_winner_6']


In [10]:
# Evaluate tipping score based on h2h
for x in [3, 4, 5, 6]:
    preds = []
    actuals = []
    for idx, row in games.iterrows():
        home_wins = row[f'h2h_wins_last{x}']
        count = row[f'h2h_count_last{x}']
        last_winner = row[f'h2h_last_winner_{x}']

        # Only predict if there is history
        if count == 0 or np.isnan(count):
            preds.append(np.nan)
            actuals.append(np.nan)
            continue

        away_wins = count - home_wins

        # Predict winner
        if home_wins > away_wins:
            pred = 1  # predict home win
        elif away_wins > home_wins:
            pred = 0  # predict away win
        else:
            # Use most recent winner: 1 if home, 0 if away
            pred = last_winner

        preds.append(pred)

        # Actual result: 1 if home won, 0 if away
        actual = 1 if row['home_result'] == 'W' else 0
        actuals.append(actual)

    preds = np.array(preds)
    actuals = np.array(actuals)
    # Only evaluate where we have a prediction
    mask = ~np.isnan(preds)
    accuracy = (preds[mask] == actuals[mask]).mean()
    print(f"Last {x} H2H games: Tip accuracy = {accuracy:.3f} (n={mask.sum()})")


Last 3 H2H games: Tip accuracy = 0.586 (n=2962)
Last 4 H2H games: Tip accuracy = 0.591 (n=2962)
Last 5 H2H games: Tip accuracy = 0.579 (n=2962)
Last 6 H2H games: Tip accuracy = 0.588 (n=2962)


In [13]:
y_true = (games['home_result'] == 'W').astype(int)
for x in [3, 4, 5, 6]:
    games[f'h2h_winrate_last{x}'] = games[f'h2h_wins_last{x}'] / games[f'h2h_count_last{x}'].replace(0, np.nan)
    games[f'h2h_winrate_last{x}'] = games[f'h2h_winrate_last{x}'].fillna(0)
for x in [3, 4, 5, 6]:
    feature = f'h2h_winrate_last{x}'
    X = games[[feature]].fillna(0)
    X = sm.add_constant(X)
    model = sm.Logit(y_true, X).fit(disp=0)
    y_pred_proba = model.predict(X)
    auc = roc_auc_score(y_true, y_pred_proba)
    print(f"X={x}: True AUC = {auc:.3f}")

X=3: True AUC = 0.613
X=4: True AUC = 0.605
X=5: True AUC = 0.605
X=6: True AUC = 0.604
